In [5]:
# -*- coding: utf-8 -*-                                   # Codificação do arquivo (acentos PT-BR)
# Script: Extração de voltas (laps) FastF1 2022–2024      # Descrição breve
# Objetivo: Usar cache SOMENTE em notebooks/fastf1_cache  # Padronização do cache

from pathlib import Path                                   # Manipulação de caminhos de forma portátil
import pandas as pd                                        # DataFrames e IO
import numpy as np                                         # Utilidades numéricas
import fastf1                                             # Biblioteca FastF1 (dados de F1)

# =========================
# Parâmetros gerais
# =========================

YEARS = [2022, 2023, 2024]                                 # Anos-alvo para extração
SKIP_IF_SAVED = True                                       # Se True, não reextrai quando já existe arquivo salvo

# =========================
# Estrutura de pastas (sempre usar notebooks/fastf1_cache)
# =========================
CWD = Path.cwd().resolve()                                 # Diretório corrente absoluto
if CWD.name.lower() == "notebooks":                        # Caso o notebook esteja dentro de /notebooks
    NB_DIR = CWD                                           # NB_DIR = .../notebooks
else:                                                      # Caso o notebook rode a partir da raiz do repositório
    NB_DIR = CWD / "notebooks"                             # NB_DIR = .../notebooks (força o uso dessa pasta)

DATA_DIR = NB_DIR / "data"                                 # Saídas e dados em notebooks/data
DATA_DIR.mkdir(parents=True, exist_ok=True)                # Garante que a pasta exista

INTERIM = DATA_DIR / "interim"                             # Subpasta para arquivos intermediários
INTERIM.mkdir(parents=True, exist_ok=True)                 # Garante que a pasta exista

CACHE_DIR = NB_DIR / "fastf1_cache"                        # **ÚNICO** cache suportado: notebooks/fastf1_cache
CACHE_DIR.mkdir(parents=True, exist_ok=True)               # Garante que a pasta exista
fastf1.Cache.enable_cache(str(CACHE_DIR))                  # Ativa cache do FastF1 nesse caminho

print(f"Cache ativo em: {CACHE_DIR}")                      # Loga caminho do cache efetivo
print(f"Saídas em:      {INTERIM}")                        # Loga caminho das saídas

# =========================
# Circuitos-alvo (mapeamento por Location e EventName)
# =========================
TARGETS = {
    # Já existentes
    "Bahrein":     {"location": ["sakhir", "bahrain"],            "event_sub": ["bahrain"]},
    "Jeddah":      {"location": ["jeddah"],                       "event_sub": ["saudi"]},
    "Australia":   {"location": ["melbourne", "australia"],       "event_sub": ["australian"]},
    "Baku":        {"location": ["baku"],                         "event_sub": ["azerbaijan"]},
    "Miami":       {"location": ["miami"],                        "event_sub": ["miami"]},
    "Monza":       {"location": ["monza"],                        "event_sub": ["italian"]},
    "Singapura":   {"location": ["singapore"],                    "event_sub": ["singapore"]},
    "Suzuka":      {"location": ["suzuka"],                       "event_sub": ["japan"]},
    "COTA":        {"location": ["austin"],                       "event_sub": ["united states"]},   # COTA/Austin (EUA)
    "Mexico":      {"location": ["mexico city"],                  "event_sub": ["mexico"]},
    "Brasil":      {"location": ["são paulo", "sao paulo"],       "event_sub": ["sao paulo", "brazil"]},
    "Abu Dhabi":   {"location": ["abu dhabi", "yas marina"],      "event_sub": ["abu dhabi"]},
    "Silverstone": {"location": ["silverstone"],                  "event_sub": ["british", "great britain"]},
    "Bélgica":     {"location": ["spa-francorchamps", "spa"],     "event_sub": ["belgian"]},
    "Hungria":     {"location": ["hungaroring", "budapest"],      "event_sub": ["hungarian"]},
    "Mônaco":      {"location": ["monaco"],                       "event_sub": ["monaco"]},
}

# =========================
# Carrega calendários (sem testes)
# =========================
all_events = []                                             # Buffer para todos os anos

for y in YEARS:                                             # Itera 2022, 2023, 2024
    cal = fastf1.get_event_schedule(y, include_testing=False).copy()  # Busca calendário e copia
    keep_cols = [c for c in ["RoundNumber","EventName","OfficialEventName","EventDate","Location","EventFormat"] if c in cal.columns]
                                                              # Mantém apenas colunas relevantes (as que existirem)
    cal = cal[keep_cols]                                    # Aplica o filtro de colunas
    cal["Year"] = y                                         # Marca o ano
    cal["loc_lc"] = cal["Location"].fillna("").str.lower()  # Normaliza Location (minúsculas) p/ matching
    cal["ename_lc"] = cal["EventName"].fillna("").str.lower()  # Normaliza EventName (minúsculas) p/ matching
    all_events.append(cal)                                  # Acumula esse calendário

events_df = pd.concat(all_events, ignore_index=True)        # Concatena calendários dos 3 anos em um DF

# =========================
# Função: regra de matching (linha do calendário ↔ circuito)
# =========================
def row_matches_target(row, target) -> bool:                # Retorna True se a linha pertence ao circuito-alvo
    loc = row["loc_lc"]                                     # Texto de Location em minúsculas
    enm = row["ename_lc"]                                   # Texto de EventName em minúsculas
    if any(sub in loc for sub in target["location"]):       # Bate por fragmentos na Location
        return True                                         # Se achou, casa
    if any(sub in enm for sub in target["event_sub"]):      # Ou bate por fragmentos no EventName
        return True                                         # Se achou, casa
    return False                                            # Caso contrário, não casa

# =========================
# Função: extrai voltas da corrida (session = 'R')
# =========================
def extract_event_race_laps(year: int, round_number: int | None, event_name: str | None) -> pd.DataFrame:
    """Carrega a sessão 'R' (Race) do evento e retorna o DataFrame de voltas (laps) com metadados."""
    if pd.notna(round_number):                              # Se tiver RoundNumber (mais robusto)
        ses = fastf1.get_session(int(year), int(round_number), 'R')  # Pega pela etapa
    else:                                                   # Senão, tenta pelo nome do evento
        ses = fastf1.get_session(int(year), str(event_name), 'R')    # (menos robusto, mas funciona)

    ses.load()                                              # Carrega dados (usa/gera cache em notebooks/fastf1_cache)
    laps = ses.laps.copy()                                  # Copia o DF de voltas

    laps["Year"] = year                                     # Marca o ano
    laps["RoundNumber"] = int(round_number) if pd.notna(round_number) else np.nan  # Marca a etapa (ou NaN)
    laps["EventName"] = ses.event.EventName if hasattr(ses, "event") else event_name  # Nome oficial do evento
    laps["SessionName"] = "Race"                            # Indica que é corrida (não Sprint/Quali)
    return laps                                             # Retorna DF

# =========================
# Loop principal: para cada circuito, extrair/ler e salvar
# =========================
all_circuits_laps = []                                      # Para montar o dataset mestre (todos circuitos)

for circuito, matchers in TARGETS.items():                  # Itera pelos 16 circuitos
    safe_circuit = circuito.lower().replace(" ", "_")       # Nome “seguro” para arquivo
    out_csv  = INTERIM / f"{safe_circuit}_2022-2024_all_laps.csv"      # Caminho CSV de saída
    out_parq = INTERIM / f"{safe_circuit}_2022-2024_all_laps.parquet"  # Caminho Parquet de saída

    # --- ETAPA 0: pular se já existe (não chama API) ---
    if SKIP_IF_SAVED and (out_parq.exists() or out_csv.exists()):      # Se já existe saída salva
        try:                                                           # Tenta ler
            if out_parq.exists():                                      # Prefere Parquet (rápido/leve)
                laps_circuito = pd.read_parquet(out_parq)              # Lê Parquet
                print(f"[PULADO] {circuito}: carregado de Parquet existente.")  # Log
            else:                                                      # Senão, lê CSV
                laps_circuito = pd.read_csv(out_csv)                   # Lê CSV
                print(f"[PULADO] {circuito}: carregado de CSV existente.")      # Log
            all_circuits_laps.append(laps_circuito)                    # Acumula no mestre
            continue                                                   # Vai ao próximo circuito
        except Exception as e:                                         # Se falhar leitura
            print(f"[AVISO] Falha ao ler saída existente de '{circuito}' ({e}). Reextraindo...")  # Avisa e segue

    # --- ETAPA 1: localizar eventos do circuito no calendário ---
    mask = events_df.apply(lambda r: row_matches_target(r, matchers), axis=1)  # Aplica matching linha a linha
    cal_hits = events_df[mask].copy().sort_values(["Year","RoundNumber","EventDate"])  # Eventos encontrados ordenados

    if cal_hits.empty:                                                # Se não achou nada
        print(f"[AVISO] Nenhum evento encontrado para '{circuito}' nos anos {YEARS}.")  # Loga aviso
        continue                                                      # Próximo circuito

    print(f"\n=== {circuito}: {len(cal_hits)} evento(s) 2022–2024 ===")  # Log informativo
    per_circuit_laps = []                                             # Buffer de voltas desse circuito
    errors = []                                                       # Buffer de erros (se houver)

    # --- ETAPA 2: para cada evento (ano), extrair voltas da corrida ---
    for _, ev in cal_hits.iterrows():                                 # Itera pelos eventos localizados
        y = int(ev["Year"])                                           # Ano do evento
        rnd = ev["RoundNumber"] if "RoundNumber" in ev and pd.notna(ev["RoundNumber"]) else None  # Round (ou None)
        ename = ev["EventName"]                                       # Nome do evento (fallback/log)

        try:                                                          # Tenta extrair laps
            df_laps = extract_event_race_laps(y, rnd, ename)          # Extrai voltas da sessão 'R'
            df_laps["Circuito"] = circuito                            # Marca circuito
            df_laps["EventDate"] = pd.to_datetime(ev["EventDate"]).date() if pd.notna(ev["EventDate"]) else pd.NaT
                                                                       # Data do evento como date
            df_laps["Location"] = ev["Location"]                      # Cidade/pista (Location)
            df_laps["OfficialEventName"] = ev.get("OfficialEventName", np.nan)  # Nome oficial completo
            per_circuit_laps.append(df_laps)                          # Acumula
            print(f"[OK] {circuito} {y} (Round {rnd if rnd is not None else 'n/a'}): {len(df_laps)} voltas.")
        except Exception as e:                                        # Trata erro
            print(f"[ERRO] {circuito} {y} (Round {rnd if rnd is not None else 'n/a'}): {e}")
            errors.append((circuito, y, str(e)))                      # Guarda info do erro

    # --- ETAPA 3: consolidar e salvar saídas do circuito ---
    if per_circuit_laps:                                              # Se coletou algo
        laps_circuito = pd.concat(per_circuit_laps, ignore_index=True)  # Concatena anos do circuito
        sort_cols = [c for c in ["Year","Driver","LapNumber"] if c in laps_circuito.columns]  # Colunas p/ ordenação
        if sort_cols:                                                 # Se existirem
            laps_circuito = laps_circuito.sort_values(sort_cols).reset_index(drop=True)  # Ordena

        laps_circuito.to_csv(out_csv, index=False, encoding="utf-8")  # Salva CSV
        laps_circuito.to_parquet(out_parq, index=False)               # Salva Parquet
        print(f"[SALVO] {circuito}:")                                  # Log de salvamento
        print(f"        CSV:     {out_csv}")                           # Caminho CSV
        print(f"        Parquet: {out_parq}")                          # Caminho Parquet

        all_circuits_laps.append(laps_circuito)                        # Acumula no mestre
    else:                                                              # Se nada foi extraído
        print(f"[AVISO] Nenhuma volta consolidada para '{circuito}'. Erros: {len(errors)}")  # Loga aviso

# =========================
# Dataset mestre (todos circuitos juntos)
# =========================
if all_circuits_laps:                                                 # Se houve algum sucesso
    master = pd.concat(all_circuits_laps, ignore_index=True)          # Concatena tudo

    preview_cols = [c for c in ["Circuito","Year","EventName","Driver","LapNumber","LapTime","Compound","Stint","PitInTime","PitOutTime"] if c in master.columns]
                                                                       # Colunas sugeridas para preview
    print("\n=== AMOSTRA (dataset mestre) ===")                       # Título do preview
    if preview_cols:                                                  # Se colunas existirem
        print(master[preview_cols].head(12).to_string(index=False))   # Mostra 12 linhas
    else:                                                             # Caso contrário
        print(master.head(12).to_string(index=False))                 # Mostra 12 linhas genéricas

    master_csv  = INTERIM / "ALLCIRCUITS_2022-2024_all_laps.csv"      # Caminho CSV mestre
    master_parq = INTERIM / "ALLCIRCUITS_2022-2024_all_laps.parquet"  # Caminho Parquet mestre
    master.to_csv(master_csv, index=False, encoding="utf-8")          # Salva CSV mestre
    master.to_parquet(master_parq, index=False)                       # Salva Parquet mestre

    print(f"\n[SALVO] Dataset mestre:")                               # Confirmação
    print(f"        CSV:     {master_csv}")                           # Caminho CSV
    print(f"        Parquet: {master_parq}")                          # Caminho Parquet
else:                                                                  # Se nenhum circuito produziu dados
    raise RuntimeError("Nenhuma volta foi extraída para os circuitos solicitados.")  # Erro explícito


Cache ativo em: C:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\notebooks\fastf1_cache
Saídas em:      C:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\notebooks\data\interim
[PULADO] Bahrein: carregado de Parquet existente.
[PULADO] Jeddah: carregado de Parquet existente.
[PULADO] Australia: carregado de Parquet existente.
[PULADO] Baku: carregado de Parquet existente.
[PULADO] Miami: carregado de Parquet existente.
[PULADO] Monza: carregado de Parquet existente.
[PULADO] Singapura: carregado de Parquet existente.
[PULADO] Suzuka: carregado de Parquet existente.
[PULADO] COTA: carregado de Parquet existente.
[PULADO] Mexico: carregado de Parquet existente.
[PULADO] Brasil: carregado de Parquet existente.
[PULADO] Abu Dhabi: carregado de Parquet existente.
[PULADO] Silverstone: carregado de Parquet existente.
[PULADO] Bélgica: carregado de Parquet existente.
[PULADO] Hungria: carregad

In [31]:
# === CÉLULA: Pós-processamento do DATASET MESTRE ===
from pathlib import Path
import pandas as pd

# Caminhos (usa o mesmo layout do script anterior; ajusta se necessário)
NB_DIR = Path.cwd().resolve() if Path.cwd().name.lower() == "notebooks" else Path.cwd().resolve() / "notebooks"
INTERIM = NB_DIR / "data" / "interim"
master_parq = INTERIM / "ALLCIRCUITS_2022-2024_all_laps.parquet"
master_csv  = INTERIM / "ALLCIRCUITS_2022-2024_all_laps.csv"

# --- Carrega o dataset mestre (prefere Parquet) ---
if master_parq.exists():
    df = pd.read_parquet(master_parq)
    print(f"[LOAD] {master_parq}")
elif master_csv.exists():
    df = pd.read_csv(master_csv)
    print(f"[LOAD] {master_csv}")
else:
    raise FileNotFoundError("Gere primeiro o dataset mestre (ALLCIRCUITS_2022-2024_all_laps.*).")

# --- Cria/normaliza LapTimeSec (segundos) ---
if "LapTimeSec" not in df.columns:
    if "LapTime" in df.columns:
        if pd.api.types.is_timedelta64_dtype(df["LapTime"]):
            df["LapTimeSec"] = df["LapTime"].dt.total_seconds()     # converte timedelta -> segundos
        else:
            df["LapTimeSec"] = pd.to_numeric(df["LapTime"], errors="coerce")  # tenta converter numérico
    else:
        df["LapTimeSec"] = pd.NA

# --- Padroniza Compound e TrackStatus para evitar perdas no filtro ---
if "Compound" in df.columns:
    df["Compound"] = df["Compound"].astype(str).str.upper()
if "TrackStatus" in df.columns:
    df["_TrackStatusStr"] = df["TrackStatus"].astype(str).str.strip()

# ========== FILTROS (ORDEM OTIMIZADA) ==========

# 0) Remover voltas marcadas como deletadas (mantém apenas Deleted == False)
if "Deleted" in df.columns:
    before = len(df)
    df = df[df["Deleted"] == False]
    after = len(df)
    print(f"Removidas {before - after} voltas com Deleted diferente de False. Restantes: {after}")
else:
    print("Coluna 'Deleted' não encontrada no DataFrame.")

# 1) Remover voltas imprecisas (IsAccurate == False)
if "IsAccurate" in df.columns:
    before = len(df)
    df = df[df["IsAccurate"]]
    after = len(df)
    print(f"Removidas {before - after} voltas com IsAccurate = False. Restantes: {after}")
else:
    print("Coluna 'IsAccurate' não encontrada no DataFrame.")

# 2) Remover linhas com tempos/setores críticos ausentes (nulos)

if "Sector1SessionTime" in df.columns:
    before = len(df)
    df = df.dropna(subset=["Sector1SessionTime"])
    after = len(df)
    print(f"Removidas {before - after} voltas com Sector1SessionTime nulo. Restantes: {after}")
else:
    print("Coluna 'Sector1SessionTime' não encontrada no DataFrame.")

if "SpeedFL" in df.columns:
    before = len(df)
    df = df.dropna(subset=["SpeedFL"])
    after = len(df)
    print(f"Removidas {before - after} voltas com SpeedFL nulo. Restantes: {after}")
else:
    print("Coluna 'SpeedFL' não encontrada no DataFrame.")

if "TyreLife" in df.columns:
    before = len(df)
    df = df.dropna(subset=["TyreLife"])
    after = len(df)
    print(f"Removidas {before - after} voltas sem TyreLife. Restantes: {after}")
else:
    print("Coluna 'TyreLife' não encontrada no DataFrame.")

# 4) Apenas compostos secos (SOFT/MEDIUM/HARD)
if "Compound" in df.columns:
    before = len(df)
    # se necessário, padronize: df["Compound"] = df["Compound"].astype(str).str.upper()
    df = df[df["Compound"].isin(["SOFT", "MEDIUM", "HARD"])]
    after = len(df)
    print(f"Removidas {before - after} voltas com pneus intermediários/molhados. Restantes: {after}")

# 5) Apenas pista verde (TrackStatus == 1)
if "TrackStatus" in df.columns:
    before = len(df)
    # auxiliar robusto para comparar com '1'
    _ts = df["TrackStatus"].astype(str).str.strip()
    mask_green = (_ts == "1") | (pd.to_numeric(df["TrackStatus"], errors="coerce") == 1)
    df = df[mask_green]
    after = len(df)
    print(f"Removidas {before - after} voltas com bandeiras/SC/VSC. Restantes: {after}")

# 6) ≤ 120% da mediana de LapTimeSec específica de cada evento (Year + EventName)
if {"LapTimeSec","Year","EventName"} <= set(df.columns):
    before_total = len(df)

    def _filter_event(g):
        med = pd.to_numeric(g["LapTimeSec"], errors="coerce").median(skipna=True)
        if pd.notna(med):
            lim = 1.2 * med
            return g[pd.to_numeric(g["LapTimeSec"], errors="coerce") <= lim]
        return g

    df = df.groupby(["Year", "EventName"], group_keys=False).apply(_filter_event)
    after_total = len(df)
    print(f"Filtradas {before_total - after_total} voltas > 120% da mediana por evento. Restantes: {after_total}")
else:
    print("Aviso: faltam colunas para o filtro de 120% por evento (LapTimeSec/Year/EventName).")


# --- Inspeção rápida ---
print("\n=== INFO PÓS-FILTRO ===")
df.info()
display(df.head())

# --- Salva dataset filtrado ---
out_parq = INTERIM / "ALLCIRCUITS_2022-2024_all_laps_FILTERED.parquet"
out_csv  = INTERIM / "ALLCIRCUITS_2022-2024_all_laps_FILTERED.csv"
df.to_parquet(out_parq, index=False)
df.to_csv(out_csv, index=False, encoding="utf-8")
print(f"\n[SALVO] Parquet: {out_parq}")
print(f"[SALVO] CSV:     {out_csv}")


[LOAD] C:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\notebooks\data\interim\ALLCIRCUITS_2022-2024_all_laps.parquet
Removidas 1856 voltas com Deleted diferente de False. Restantes: 47392
Removidas 6364 voltas com IsAccurate = False. Restantes: 41028
Removidas 94 voltas com Sector1SessionTime nulo. Restantes: 40934
Removidas 3 voltas com SpeedFL nulo. Restantes: 40931
Removidas 87 voltas sem TyreLife. Restantes: 40844
Removidas 2609 voltas com pneus intermediários/molhados. Restantes: 38235
Removidas 1008 voltas com bandeiras/SC/VSC. Restantes: 37227
Filtradas 60 voltas > 120% da mediana por evento. Restantes: 37167

=== INFO PÓS-FILTRO ===
<class 'pandas.core.frame.DataFrame'>
Index: 37167 entries, 32823 to 26657
Data columns (total 41 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   Time                37167 non-null  timedelta64[ns]
 1   Driver         

C:\Users\pedro\AppData\Local\Temp\ipykernel_7364\2902129792.py:112: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(["Year", "EventName"], group_keys=False).apply(_filter_event)


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,Year,RoundNumber,EventName,SessionName,Circuito,EventDate,Location,OfficialEventName,LapTimeSec,_TrackStatusStr
32823,0 days 01:05:25.360000,ALB,23,0 days 00:01:33.227000,2.0,1.0,NaT,NaT,0 days 00:00:18.788000,0 days 00:00:39.586000,...,2022,22,Abu Dhabi Grand Prix,Race,Abu Dhabi,2022-11-20,Yas Island,FORMULA 1 ETIHAD AIRWAYS ABU DHABI GRAND PRIX ...,93.227,1
32824,0 days 01:06:57.716000,ALB,23,0 days 00:01:32.356000,3.0,1.0,NaT,NaT,0 days 00:00:18.723000,0 days 00:00:38.407000,...,2022,22,Abu Dhabi Grand Prix,Race,Abu Dhabi,2022-11-20,Yas Island,FORMULA 1 ETIHAD AIRWAYS ABU DHABI GRAND PRIX ...,92.356,1
32825,0 days 01:08:30.853000,ALB,23,0 days 00:01:33.137000,4.0,1.0,NaT,NaT,0 days 00:00:18.687000,0 days 00:00:38.424000,...,2022,22,Abu Dhabi Grand Prix,Race,Abu Dhabi,2022-11-20,Yas Island,FORMULA 1 ETIHAD AIRWAYS ABU DHABI GRAND PRIX ...,93.137,1
32826,0 days 01:10:03.505000,ALB,23,0 days 00:01:32.652000,5.0,1.0,NaT,NaT,0 days 00:00:18.785000,0 days 00:00:38.870000,...,2022,22,Abu Dhabi Grand Prix,Race,Abu Dhabi,2022-11-20,Yas Island,FORMULA 1 ETIHAD AIRWAYS ABU DHABI GRAND PRIX ...,92.652,1
32827,0 days 01:11:35.807000,ALB,23,0 days 00:01:32.302000,6.0,1.0,NaT,NaT,0 days 00:00:18.718000,0 days 00:00:38.967000,...,2022,22,Abu Dhabi Grand Prix,Race,Abu Dhabi,2022-11-20,Yas Island,FORMULA 1 ETIHAD AIRWAYS ABU DHABI GRAND PRIX ...,92.302,1



[SALVO] Parquet: C:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\notebooks\data\interim\ALLCIRCUITS_2022-2024_all_laps_FILTERED.parquet
[SALVO] CSV:     C:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\notebooks\data\interim\ALLCIRCUITS_2022-2024_all_laps_FILTERED.csv


In [32]:
# === CÉLULA: Resumo estatístico e informativo do DataFrame final ===

print("\n=== RESUMO GERAL ===")
print(f"Total de linhas: {len(df)}")
print(f"Total de colunas: {len(df.columns)}")

# --- Visão geral do DataFrame ---
print("\n=== INFO ===")
df.info()

# --- Estatísticas descritivas para colunas numéricas ---
print("\n=== DESCRIBE (Numéricas) ===")
display(df.describe().T)  # .T para facilitar leitura (variáveis em linhas)

# --- Estatísticas descritivas também para colunas de texto/categóricas ---
print("\n=== DESCRIBE (Categóricas) ===")
display(df.describe(include='object').T)

# --- Frequência de valores para colunas não numéricas ---
print("\n=== FREQUÊNCIA DE VALORES (TOP 5) ===")
for col in df.columns:
    if df[col].dtype == 'object' or df[col].dtype.name == 'category' or df[col].dtype == 'bool':
        print(f"\nColuna: {col}")
        print(df[col].value_counts(dropna=False).head(5))



=== RESUMO GERAL ===
Total de linhas: 37167
Total de colunas: 41

=== INFO ===
<class 'pandas.core.frame.DataFrame'>
Index: 37167 entries, 32823 to 26657
Data columns (total 41 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   Time                37167 non-null  timedelta64[ns]
 1   Driver              37167 non-null  object         
 2   DriverNumber        37167 non-null  object         
 3   LapTime             37167 non-null  timedelta64[ns]
 4   LapNumber           37167 non-null  float64        
 5   Stint               37167 non-null  float64        
 6   PitOutTime          0 non-null      timedelta64[ns]
 7   PitInTime           0 non-null      timedelta64[ns]
 8   Sector1Time         37167 non-null  timedelta64[ns]
 9   Sector2Time         37167 non-null  timedelta64[ns]
 10  Sector3Time         37167 non-null  timedelta64[ns]
 11  Sector1SessionTime  37167 non-null  timedelta64[ns]
 12  Sector2Se

,count,mean,min,25%,50%,75%,max,std
Time,37167,0 days 01:54:04.129337880,0 days 00:57:55.986000,0 days 01:28:44.879500,0 days 01:52:30.998000,0 days 02:14:54.689000,0 days 04:11:49.536000,0 days 00:32:21.027943431
LapTime,37167,0 days 00:01:32.641900852,0 days 00:01:12.486000,0 days 00:01:24.600000,0 days 00:01:32.655000,0 days 00:01:39.003000,0 days 00:02:09.869000,0 days 00:00:09.865682605
LapNumber,37167.0,29.938817,2.0,16.0,29.0,43.0,78.0,16.93494
Stint,37167.0,2.042834,1.0,1.0,2.0,3.0,6.0,0.887618
PitOutTime,0,NaT,NaT,NaT,NaT,NaT,NaT,NaT
PitInTime,0,NaT,NaT,NaT,NaT,NaT,NaT,NaT
Sector1Time,37167,0 days 00:00:28.912301046,0 days 00:00:17.257000,0 days 00:00:27.621500,0 days 00:00:29.697000,0 days 00:00:31.685000,0 days 00:00:44.188000,0 days 00:00:05.511908669
Sector2Time,37167,0 days 00:00:36.742134205,0 days 00:00:17.488000,0 days 00:00:30.776000,0 days 00:00:38.094000,0 days 00:00:42.106000,0 days 00:00:54.820000,0 days 00:00:07.353032698
Sector3Time,37167,0 days 00:00:26.987465601,0 days 00:00:16.913000,0 days 00:00:23.823000,0 days 00:00:26.008000,0 days 00:00:30.290000,0 days 00:00:47.288000,0 days 00:00:05.103700616
Sector1SessionTime,37167,0 days 01:53:00.434537466,0 days 00:57:01.888000,0 days 01:27:39.593000,0 days 01:51:26.588000,0 days 02:13:51.341500,0 days 04:10:28.032000,0 days 00:32:21.485946686



=== DESCRIBE (Categóricas) ===


,count,unique,top,freq
Driver,37167,28,NOR,2004
DriverNumber,37167,31,4,2004
IsPersonalBest,37167,2,False,28584
Compound,37167,3,HARD,19050
Team,37167,12,Mercedes,3878
TrackStatus,37167,1,1,37167
Deleted,37167,1,False,37167
DeletedReason,37167,1,,37167
EventName,37167,16,Hungarian Grand Prix,3471
SessionName,37167,1,Race,37167



=== FREQUÊNCIA DE VALORES (TOP 5) ===

Coluna: Driver
Driver
NOR    2004
VER    1990
ALO    1973
RUS    1955
HAM    1923
Name: count, dtype: int64

Coluna: DriverNumber
DriverNumber
4     2004
1     1990
14    1973
63    1955
44    1923
Name: count, dtype: int64

Coluna: IsPersonalBest
IsPersonalBest
False    28584
True      8583
Name: count, dtype: int64

Coluna: Compound
Compound
HARD      19050
MEDIUM    13439
SOFT       4678
Name: count, dtype: int64

Coluna: FreshTyre
FreshTyre
True     29341
False     7826
Name: count, dtype: int64

Coluna: Team
Team
Mercedes           3878
McLaren            3862
Red Bull Racing    3858
Ferrari            3832
Aston Martin       3777
Name: count, dtype: int64

Coluna: TrackStatus
TrackStatus
1    37167
Name: count, dtype: int64

Coluna: Deleted
Deleted
False    37167
Name: count, dtype: int64

Coluna: DeletedReason
DeletedReason
    37167
Name: count, dtype: int64

Coluna: FastF1Generated
FastF1Generated
False    37167
Name: count, dtype: int64

In [33]:
# Garante que temos as colunas necessárias
if {"Year", "EventName", "Driver", "LapNumber"} <= set(df.columns):
    # Conta quantas voltas cada piloto completou em cada evento
    laps_por_piloto = df.groupby(["Year", "EventName", "Driver"]).size().reset_index(name="Voltas")

    # Agora agrupa por evento e calcula média e mediana do número de voltas
    resumo_evento = laps_por_piloto.groupby([ "EventName", "Year"])["Voltas"].agg(
        media_voltas="mean",
        mediana_voltas="median"
    ).reset_index()

    print("\n=== Média e Mediana de Voltas por Evento ===")
    display(resumo_evento)

else:
    print("O DataFrame precisa ter as colunas: Year, EventName, Driver e LapNumber.")


=== Média e Mediana de Voltas por Evento ===


,EventName,Year,media_voltas,mediana_voltas
0,Abu Dhabi Grand Prix,2022,49.450000,50.5
1,Abu Dhabi Grand Prix,2023,51.450000,52.0
2,Abu Dhabi Grand Prix,2024,44.947368,47.0
3,Australian Grand Prix,2022,41.684211,44.0
4,Australian Grand Prix,2023,39.578947,44.0
5,Australian Grand Prix,2024,44.578947,49.0
6,Azerbaijan Grand Prix,2022,36.600000,41.0
7,Azerbaijan Grand Prix,2023,40.100000,42.0
8,Azerbaijan Grand Prix,2024,42.500000,44.5
9,Bahrain Grand Prix,2022,42.600000,42.0
